# QariOCR Fine-Tuning with Unsloth
## Enhanced Quran Verification - KDN Compliant

**Modified from**: [Official Unsloth Qwen2.5-VL Notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen2.5_VL_(7B)-Vision.ipynb)

**Dataset**: 1,128 samples (604 perfect + 400 synthetic errors + 124 KDN examples)  
**Model**: Qwen2-VL-2B-Instruct (optimized for free T4 GPU)  
**Training Time**: .... on free Google Colab  

---

### 📋 Before Starting:

1. **Enable GPU**: Runtime → Change runtime type → GPU (T4)
2. **Upload to Google Drive**:
   - `train_enhanced.json` → `My Drive/QariOCR_Training/`
   - `val_enhanced.json` → `My Drive/QariOCR_Training/`
   - `../database/reference_imgs/` (604 images) → `My Drive/QariOCR_Training/../database/reference_imgs/`
   - `../database/extracted_examples/` (124 images) → `My Drive/QariOCR_Training/../database/extracted_examples/`

3. **Run**: Press Runtime → Run all

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastVisionModel
import torch

# Load Qwen2-VL-2B model (optimized for free T4 GPU)
# Using 2B instead of 7B for:
# - Fits on free Colab T4 (15 GB VRAM)
# - Faster training (4-6 hours vs 8-12 hours)
# - Still excellent accuracy for QariOCR

print("🔄 Loading Qwen2-VL-2B-Instruct...")
print("   This takes ~5 minutes (downloading 4GB model)...\n")

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",  # 2B model, 4-bit quantized
    load_in_4bit = True,  # Use 4-bit to reduce memory
    use_gradient_checkpointing = "unsloth",  # Memory efficient checkpointing
)

print("✅ Loaded Qwen2-VL-2B-Instruct")
print(f"   Model: {model.config.model_type}")
print(f"   Device: {next(model.parameters()).device}")
print(f"   Memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Note: Other available vision models include Llama 3.2 Vision, Pixtral, etc.
# See: https://huggingface.co/unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.1: Fast Qwen2_5_Vl patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.90G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

<a name="Data"></a>### Data Prep - QariOCR Enhanced DatasetWe're using the **QariOCR Enhanced Dataset** with 1,128 samples:- **604 Perfect Uthmani Text** (53.5%) - Clean Quran pages for text extraction- **400 Synthetic Errors** (35.5%) - Programmatically generated errors across 39 error types- **124 KDN Official Examples** (11.0%) - Real audit examples from official guidelinesThe model will learn to:1. Extract Uthmani text with high accuracy2. Detect and classify errors (diacritics, letters, structure)3. Provide KDN-compliant verification reports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os
from datasets import Dataset
from PIL import Image

# Dataset paths in Google Drive
DATASET_PATH = "/content/drive/MyDrive/QariOCR_Training"
TRAIN_FILE = f"{DATASET_PATH}/train_enhanced.json"
VAL_FILE = f"{DATASET_PATH}/val_enhanced.json"
IMAGE_DIR = f"{DATASET_PATH}/database/reference_imgs"
EXTRACTED_EXAMPLES_DIR = f"{DATASET_PATH}/database/extracted_examples"

print("📁 Checking dataset files...\n")

# Verify files exist
train_exists = os.path.exists(TRAIN_FILE)
val_exists = os.path.exists(VAL_FILE)
imgs_exist = os.path.exists(IMAGE_DIR)
extracted_exist = os.path.exists(EXTRACTED_EXAMPLES_DIR)

print(f"{'✓' if train_exists else '✗'} train_enhanced.json")
print(f"{'✓' if val_exists else '✗'} val_enhanced.json")
print(f"{'✓' if imgs_exist else '✗'} ../database/reference_imgs/")
print(f"{'✓' if extracted_exist else '✗'} ../database/extracted_examples/")

if imgs_exist:
    img_count = len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')])
    print(f"{'✓' if img_count == 604 else '⚠️'} Found {img_count}/604 reference images")

if extracted_exist:
    extracted_count = len([f for f in os.listdir(EXTRACTED_EXAMPLES_DIR) if f.endswith('.jpg')])
    print(f"{'✓' if extracted_count == 124 else '⚠️'} Found {extracted_count}/124 KDN examples")

if not (train_exists and val_exists and imgs_exist and extracted_exist):
    raise FileNotFoundError(
        "⚠️ Dataset files not found in Google Drive!\n"
        "Please upload to: My Drive/QariOCR_Training/\n"
        "  - train_enhanced.json\n"
        "  - val_enhanced.json\n"
        "  - ../database/reference_imgs/ (604 images)\n"
        "  - ../database/extracted_examples/ (124 images)"
    )

print("\n✅ All files found! Loading dataset...")

README.md:   0%|          | 0.00/519 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/38.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/68686 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7632 [00:00<?, ? examples/s]

Now let's load the actual dataset and convert it to the format required for training.

In [ ]:
def load_qari_dataset(json_file):
    """Load QariOCR enhanced dataset from JSON file with lazy image loading."""
    print(f"Loading {json_file}...")
    
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    formatted_samples = []
    
    for i, item in enumerate(data):
        try:
            # Extract image path from the nested structure
            image_content = item['messages'][0]['content'][0]
            img_path = image_content['image']
            
            # Adjust path to absolute path in Google Drive
            if not img_path.startswith('/'):
                img_path = os.path.join(DATASET_PATH, img_path)
            
            # Verify image exists
            if not os.path.exists(img_path):
                print(f"⚠️ Warning: Image not found: {img_path}")
                continue
            
            # Extract prompt and response
            user_text = item['messages'][0]['content'][1]['text']
            assistant_text = item['messages'][1]['content'][0]['text']
            
            # Store image path instead of loading image (lazy loading)
            formatted_samples.append({
                "image_path": img_path,  # Store path instead of image
                "user_prompt": user_text,
                "assistant_response": assistant_text,
                "metadata": item.get('metadata', {})
            })
            
            # Progress indicator for large datasets
            if (i + 1) % 100 == 0:
                print(f"   Processed {i + 1}/{len(data)} samples...")
            
        except Exception as e:
            print(f"⚠️ Error processing item {i}: {e}")
            continue
    
    print(f"✅ Loaded {len(formatted_samples)} valid samples (images will be loaded during training)")
    return formatted_samples

# Load training and validation datasets
train_samples = load_qari_dataset(TRAIN_FILE)
val_samples = load_qari_dataset(VAL_FILE)

print(f"\n📊 Dataset Summary:")
print(f"   Training: {len(train_samples)} samples")
print(f"   Validation: {len(val_samples)} samples")
print(f"   Total: {len(train_samples) + len(val_samples)} samples")

# Show a sample
if train_samples:
    sample = train_samples[0]
    print(f"\n📝 Sample Entry:")
    print(f"   Task: {sample['metadata'].get('task', 'unknown')}")
    print(f"   Has error: {sample['metadata'].get('has_error', False)}")
    print(f"   Prompt: {sample['user_prompt'][:100]}...")
    print(f"   Response: {sample['assistant_response'][:100]}...")

Dataset({
    features: ['image', 'text'],
    num_rows: 68686
})

In [ ]:
# Display a sample image from the training set
from IPython.display import display

# Check if train_samples is defined
if 'train_samples' not in locals():
    print("⚠️ ERROR: train_samples not defined!")
    print("   Please run Cell 10 (Data Loading) first to load the datasets.")
    raise NameError("train_samples not defined. Run Cell 10 first.")

if train_samples:
    # Load image on-demand for preview
    sample_image = Image.open(train_samples[0]["image_path"]).convert("RGB")
    
    print("📸 Sample Image Preview:")
    print("=" * 70)
    display(sample_image)
    print("=" * 70)
    print(f"Image size: {sample_image.size[0]} x {sample_image.size[1]} pixels")
    print(f"Mode: {sample_image.mode}")
    print(f"Format: {sample_image.format if hasattr(sample_image, 'format') else 'PIL Image'}")
    print(f"Image path: {train_samples[0]['image_path']}")
else:
    print("⚠️ No training samples loaded")

NameError: name 'train_samples' is not defined

In [ ]:
# Show the prompt and expected response
if train_samples:
    sample = train_samples[0]
    print("=" * 70)
    print("USER PROMPT:")
    print("=" * 70)
    print(sample["user_prompt"])
    print("\n" + "=" * 70)
    print("ASSISTANT RESPONSE:")
    print("=" * 70)
    print(sample["assistant_response"])
    print("=" * 70)

### Dataset Statistics

Let's look at the distribution of tasks and error types in our dataset:

In [ ]:
# Analyze dataset composition
from collections import Counter

def analyze_dataset(samples, name="Dataset"):
    """Analyze task and error distribution."""
    tasks = [s['metadata'].get('task', 'unknown') for s in samples]
    has_errors = [s['metadata'].get('has_error', False) for s in samples]
    
    task_counts = Counter(tasks)
    error_count = sum(has_errors)
    
    print(f"\n📊 {name} Analysis:")
    print(f"   Total samples: {len(samples)}")
    print(f"   With errors: {error_count} ({100*error_count/len(samples):.1f}%)")
    print(f"   Without errors: {len(samples)-error_count} ({100*(len(samples)-error_count)/len(samples):.1f}%)")
    print(f"\n   Task distribution:")
    for task, count in task_counts.most_common():
        print(f"     - {task}: {count} ({100*count/len(samples):.1f}%)")

# Check if datasets are loaded
if 'train_samples' not in locals() or 'val_samples' not in locals():
    print("⚠️ DATASET NOT LOADED!")
    print("   Please run Cell 10 (Data Loading) first to load the datasets.")
    print("   This cell analyzes the loaded dataset composition.")
else:
    analyze_dataset(train_samples, "Training Set")
    analyze_dataset(val_samples, "Validation Set")

To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "user",
  "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
},
{ "role": "assistant",
  "content": [{"type": "text",  "text": A} ]
},
]
```

In [ ]:
def convert_to_conversation(sample):
    """
    Convert QariOCR sample to Unsloth conversation format.
    
    Our enhanced dataset has:
    - image_path: Path to image (loaded on-demand)
    - user_prompt: The instruction/prompt text
    - assistant_response: The expected output (text or error report)
    - metadata: Task info, error flags, etc.
    """
    try:
        # Load image on-demand to save memory
        image = Image.open(sample["image_path"]).convert("RGB")
        
        conversation = [
            { 
                "role": "user",
                "content": [
                    {"type": "text", "text": sample["user_prompt"]},
                    {"type": "image", "image": image}
                ]
            },
            { 
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["assistant_response"]}
                ]
            },
        ]
        return {"messages": conversation}
    except Exception as e:
        print(f"⚠️ Error loading image {sample.get('image_path', 'unknown')}: {e}")
        return None

print("✅ Conversation converter function defined")

### ⚠️ IMPORTANT: Execution Order

**Before running the conversion cell below, make sure you have:**

1. ✅ **Run Cell 10** (Data Loading) - This loads `train_samples` and `val_samples`
2. ✅ **Wait for completion** - Should take ~30-60 seconds
3. ✅ **Check for success** - Look for "✅ Loaded X valid samples" message

**If you see an error about `train_samples` not defined:**
- Go back to Cell 10 and run it first
- Don't skip ahead to this cell until Cell 10 completes successfully


### Convert datasets to conversation format

Now let's convert both training and validation sets into the format required by Unsloth:

In [ ]:
from datasets import Dataset

# Check if datasets are loaded
if 'train_samples' not in locals() or 'val_samples' not in locals():
    print("⚠️ DATASET NOT LOADED!")
    print("   Please run Cell 10 (Data Loading) first to load the datasets.")
    print("   This cell loads the JSON data and creates the sample lists.")
    print("\n📋 To fix this:")
    print("   1. Go to Cell 10 (Data Loading)")
    print("   2. Click 'Run' or press Shift+Enter")
    print("   3. Wait for it to complete (should take ~30-60 seconds)")
    print("   4. Then come back to this cell")
    print("\n⏸️  This cell will wait for you to load the data first...")
    # Skip the rest of the cell
    train_samples = None
    val_samples = None

if train_samples is None or val_samples is None:
    print("⏸️  Skipping dataset conversion - please load data first.")
elif not train_samples or not val_samples:
    print("⚠️ EMPTY DATASETS!")
    print(f"   train_samples: {len(train_samples) if train_samples else 0}")
    print(f"   val_samples: {len(val_samples) if val_samples else 0}")
    print("   Check that your JSON files contain valid data.")
    print("\n📋 To fix this:")
    print("   1. Verify train_enhanced.json and val_enhanced.json exist")
    print("   2. Check that the files are not empty")
    print("   3. Re-run Cell 10 (Data Loading)")
    print("⏸️  Skipping dataset conversion - please fix data issues first.")
else:
    print("Converting datasets to conversation format...")
    print(f"   Training samples: {len(train_samples)}")
    print(f"   Validation samples: {len(val_samples)}")

    # Convert training dataset
    print("\n🔄 Converting training dataset...")
    try:
        train_conversations = []
        failed_count = 0
        for i, sample in enumerate(train_samples):
            conversation = convert_to_conversation(sample)
            if conversation is not None:
                train_conversations.append(conversation)
            else:
                failed_count += 1
            if (i + 1) % 50 == 0:
                print(f"   Processed {i + 1}/{len(train_samples)} training samples...")
        
        converted_train_dataset = Dataset.from_list(train_conversations)
        print(f"✅ Converted {len(converted_train_dataset)} training samples")
        if failed_count > 0:
            print(f"⚠️ Skipped {failed_count} training samples due to image loading errors")
    except Exception as e:
        print(f"❌ Error converting training dataset: {e}")
        raise

    # Convert validation dataset
    print("\n🔄 Converting validation dataset...")
    try:
        val_conversations = []
        failed_count = 0
        for i, sample in enumerate(val_samples):
            conversation = convert_to_conversation(sample)
            if conversation is not None:
                val_conversations.append(conversation)
            else:
                failed_count += 1
            if (i + 1) % 50 == 0:
                print(f"   Processed {i + 1}/{len(val_samples)} validation samples...")
        
        converted_val_dataset = Dataset.from_list(val_conversations)
        print(f"✅ Converted {len(converted_val_dataset)} validation samples")
        if failed_count > 0:
            print(f"⚠️ Skipped {failed_count} validation samples due to image loading errors")
    except Exception as e:
        print(f"❌ Error converting validation dataset: {e}")
        raise

    print(f"\n📊 Final Dataset:")
    print(f"   Training: {len(converted_train_dataset)} samples")
    print(f"   Validation: {len(converted_val_dataset)} samples")
    print(f"   Type: {type(converted_train_dataset)}")
    print("✅ Dataset conversion completed successfully!")

Let's verify the conversation format is correct by examining the first training example:

In [ ]:
# Display first training example in conversation format
import pprint
pp = pprint.PrettyPrinter(indent=2, width=100)

print("First training example:")
print("=" * 80)

# Check if converted_train_dataset exists
if 'converted_train_dataset' in locals() and len(converted_train_dataset) > 0:
    example = converted_train_dataset[0]
    # Print without showing the actual image object (it's huge)
    example_to_show = {
        "messages": [
            {
                "role": example["messages"][0]["role"],
                "content": [
                    example["messages"][0]["content"][0],  # text
                    {"type": "image", "image": "<PIL.Image.Image object>"}  # placeholder
                ]
            },
            example["messages"][1]  # assistant message
        ]
    }
    
    pp.pprint(example_to_show)
else:
    print("⚠️  converted_train_dataset not available yet.")
    print("   Run the conversion cell first (Cell 18)")

### Test Model Before Training

Let's see what the base model outputs for a QariOCR example before any fine-tuning:

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

# Use a sample from our dataset
test_sample = train_samples[0]
test_image = Image.open(test_sample["image_path"]).convert("RGB")
test_instruction = test_sample["user_prompt"]

print("Testing with prompt:")
print("-" * 80)
print(test_instruction)
print("-" * 80)
print("\nModel output (before training):")
print("-" * 80)

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": test_instruction}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    test_image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs, 
    streamer=text_streamer, 
    max_new_tokens=256,  # Increased for longer Quran text
    use_cache=True, 
    temperature=0.3,  # Lower temperature for more consistent output
    min_p=0.1
)

print("-" * 80)
print("\nExpected output:")
print("-" * 80)
print(test_sample["assistant_response"][:300] + "..." if len(test_sample["assistant_response"]) > 300 else test_sample["assistant_response"])

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

We use our new `UnslothVisionDataCollator` which will help in our vision finetuning setup.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

# Calculate steps for 3 epochs
steps_per_epoch = len(converted_train_dataset) // (2 * 4)  # batch_size * grad_accum
total_steps_3_epochs = steps_per_epoch * 3
eval_steps = max(50, steps_per_epoch // 5)  # Evaluate 5 times per epoch

print(f"📊 Training Configuration:")
print(f"   Training samples: {len(converted_train_dataset)}")
print(f"   Validation samples: {len(converted_val_dataset)}")
print(f"   Steps per epoch: {steps_per_epoch}")
print(f"   Total steps (3 epochs): {total_steps_3_epochs}")
print(f"   Evaluation every: {eval_steps} steps")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_train_dataset,
    eval_dataset = converted_val_dataset,  # Added validation set
    args = SFTConfig(
        # Training
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 4,  # Effective batch size = 8
        num_train_epochs = 3,  # Full training for 3 epochs
        # max_steps = 60,  # Comment out for full training
        
        # Learning rate
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",  # Cosine decay
        warmup_steps = 50,
        
        # Optimization
        optim = "adamw_8bit",
        weight_decay = 0.01,
        max_grad_norm = 1.0,
        
        # Logging & Evaluation
        logging_steps = 10,
        eval_strategy = "steps",  # Evaluate during training
        eval_steps = eval_steps,
        save_strategy = "steps",
        save_steps = eval_steps,
        save_total_limit = 2,  # Keep only 2 best checkpoints
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        
        # Output
        output_dir = "qari_ocr_checkpoints",
        report_to = "none",  # Change to "wandb" for Weights & Biases tracking
        seed = 42,
        
        # Vision finetuning requirements (MUST HAVE):
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

print("\n✅ Trainer configured!")
print(f"   Output directory: qari_ocr_checkpoints/")
print(f"   Best model will be loaded at end based on validation loss")

Unsloth: Model does not have a default image size - using 512


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
6.836 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 68,686 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 51,521,536 of 8,343,688,192 (0.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.772000
2,3.315700
3,3.383500
4,2.292700
5,2.041700
6,2.029900
7,1.515400
8,1.002100
9,0.721900
10,0.741700


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

184.4197 seconds used for training.
3.07 minutes used for training.
Peak reserved memory = 7.568 GB.
Peak reserved memory for training = 0.732 GB.
Peak reserved memory % of max memory = 51.34 %.
Peak reserved memory for training % of max memory = 4.966 %.


<a name="Inference"></a>
### Inference After Training

Let's test the fine-tuned model on a validation sample to see the improvement!

For Quran text extraction, we use `temperature = 0.3` (lower for more deterministic output) and `max_new_tokens = 512` (for longer text).

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

# Test on a VALIDATION sample (not seen during training)
test_sample = val_samples[0]
image = Image.open(test_sample["image_path"]).convert("RGB")
instruction = test_sample["user_prompt"]

print("🧪 Testing trained model on validation sample")
print("=" * 80)
print("PROMPT:")
print(instruction)
print("\n" + "=" * 80)
print("MODEL OUTPUT (after training):")
print("=" * 80)

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs, 
    streamer=text_streamer, 
    max_new_tokens=512,  # Longer for full Quran pages
    use_cache=True, 
    temperature=0.3,  # Lower for more consistent output
    min_p=0.1
)

print("\n" + "=" * 80)
print("EXPECTED OUTPUT:")
print("=" * 80)
print(test_sample["assistant_response"][:500] + "..." if len(test_sample["assistant_response"]) > 500 else test_sample["assistant_response"])
print("=" * 80)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
# Save to Google Drive so you can download it
SAVE_PATH = "/content/drive/MyDrive/QariOCR_Trained_Model"

print(f"💾 Saving fine-tuned model to Google Drive...")
print(f"   Location: {SAVE_PATH}")

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("\n✅ Model saved successfully!")
print(f"\n📦 Saved files:")
print(f"   • LoRA adapters (~100-200 MB)")
print(f"   • Tokenizer")
print(f"   • Configuration files")
print(f"\n📥 To download:")
print(f"   1. Go to Google Drive → My Drive/QariOCR_Trained_Model/")
print(f"   2. Right-click → Download")
print(f"   3. Extract to your MacBook")

# Optional: Push to Hugging Face Hub for easy sharing/deployment
# model.push_to_hub("your_username/qari-ocr-uthmani", token="hf_...") 
# tokenizer.push_to_hub("your_username/qari-ocr-uthmani", token="hf_...")

[]

### Loading the Saved Model

If you want to load the LoRA adapters we just saved for inference (in a new session), set `False` to `True` and run:

In [ ]:
if False:  # Set to True to test loading the saved model
    print("🔄 Loading saved model from Google Drive...")
    
    from unsloth import FastVisionModel
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Load the fine-tuned model
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "/content/drive/MyDrive/QariOCR_Trained_Model",  # Your saved model
        load_in_4bit = True,
    )
    FastVisionModel.for_inference(model)
    print("✅ Model loaded!")
    
    # Test with a Quran page
    # You'll need to load your image - example:
    from PIL import Image
    image = Image.open("/content/drive/MyDrive/QariOCR_Training/../database/reference_imgs/001.jpg")
    instruction = "Extract and verify all Uthmani Quranic text from this page. Report any errors."
    
    print("\n📖 Testing on Quran page...")
    print("=" * 80)
    
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": instruction}
        ]}
    ]
    
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")
    
    from transformers import TextStreamer
    text_streamer = TextStreamer(tokenizer, skip_prompt=True)
    _ = model.generate(
        **inputs, 
        streamer=text_streamer, 
        max_new_tokens=512,
        use_cache=True, 
        temperature=0.3, 
        min_p=0.1
    )
    print("=" * 80)

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit
if False: model.save_pretrained_merged("unsloth_finetune", tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", tokenizer, token = "PUT_HERE")